# Experiment 01: Metric Sanity

Artificial subspace sanity checks for the new full/top alignment metrics. This notebook is not executed by default.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "config.py").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate repository root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")


In [ ]:
import numpy as np
import pandas as pd

from src.metrics import (
    build_sampled_basis_matrix_from_indices,
    calculate_subspace_alignment_from_matrices,
)


In [ ]:
NUM_FREQS = 4
SEQ_LEN = 2048
TARGET_INDICES = np.arange(SEQ_LEN, dtype=np.int64)
THETAS = tuple(np.linspace(0.10 * np.pi, 0.70 * np.pi, NUM_FREQS))
ORTH_NOISE_SCALES = [0.05, 0.10, 0.20, 0.50]

F = build_sampled_basis_matrix_from_indices(
    time_mode="discrete",
    target_indices=TARGET_INDICES,
    thetas=THETAS,
)
Qf, _ = np.linalg.qr(F)
P_perp = np.eye(Qf.shape[0]) - Qf @ Qf.T
rng = np.random.default_rng(42)


In [ ]:
def random_full_row_rank_matrix(rows: int, cols: int, seed: int) -> np.ndarray:
    rng_local = np.random.default_rng(seed)
    return rng_local.standard_normal((rows, cols))


def make_cases(alpha: float) -> dict[str, np.ndarray]:
    A = random_full_row_rank_matrix(Qf.shape[1], Qf.shape[1], seed=123)
    Z = rng.standard_normal((Qf.shape[0], Qf.shape[1]))
    Z_perp = P_perp @ Z
    return {
        "same_subspace": Qf @ A,
        "same_plus_orth_noise": np.concatenate([Qf @ A, alpha * Z_perp], axis=1),
        "random": rng.standard_normal((Qf.shape[0], Qf.shape[1])),
        "random_perp": P_perp @ rng.standard_normal((Qf.shape[0], Qf.shape[1])),
    }


In [ ]:
records = []
for alpha in ORTH_NOISE_SCALES:
    for case_name, H in make_cases(alpha).items():
        metrics = calculate_subspace_alignment_from_matrices(H=H, F=F, theory_dim=2 * NUM_FREQS)
        records.append({
            "alpha": alpha,
            "case": case_name,
            "align_coverage_full": metrics["align_coverage_full"],
            "align_purity_full": metrics["align_purity_full"],
            "align_coverage_top": metrics["align_coverage_top"],
            "align_mean_angle_deg_top": metrics["align_mean_angle_deg_top"],
            "recon_r2_qf_from_h": metrics["recon_r2_qf_from_h"],
        })

results_df = pd.DataFrame(records)
results_df.sort_values(["alpha", "case"]).reset_index(drop=True)
